<a href="https://colab.research.google.com/github/pierrenotfound77/PLMezSeq/blob/main/PLMezSeq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**PLMezSeq: ESM-based Tool for Quickly Processing Protein Sequence with Non-standard Amino Acids for PLMs**

<font color="gray">*For Protein Language Models (PLMs) that do not process non-standard amino acids in sequences, this tool helps replace those with most-likely standard ones that are predicted by ESM2.*</font>

Input: a csv file containing two columns of data: protein sequence and label OR a single protein string

Output: a csv file process protein sequence with all non-standard amino acids converted to 20 standard ones.

##**Step1: Upload Your .csv File (Optional)**

Make sure you have included a <font color="red">**"sequence"**</font> column in the csv file :)

In [ ]:
# @title Click the Run Button to upload.
from google.colab import files
import io
import pandas as pd

try:
  # Upload a file from your local machine
  uploaded = files.upload()

  # Assuming the user has uploaded a single CSV file.
  uploaded_filename = None
  for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))
    uploaded_filename = fn # Save the filename
    # Read the CSV file into a DataFrame
    df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))

  # Display the first few rows of the DataFrame
  display(df.head())

  print("You have successfully uploaded " + uploaded_filename)

except:
  print("Upload canceled")

##**Step2: Initialize ESM2**

In [2]:
# @title Click the Run Button to initialize the model and device.
import torch
from transformers import AutoTokenizer, EsmForMaskedLM

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

#print("Using device:", device)

# Load ESM2 model
model_name = "facebook/esm2_t33_650M_UR50D"

# Set tokenizer and MLM model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)

model = model.to(device)

# table for standard amino acids
standard_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.61GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/539 [00:00<?, ?it/s]

##**Step3: Process the Sequence**

You can either choose the first module to predict for a single string, or choose the second module to run you sequences in your .csv file in batches.

In [12]:
# @title Predict Non-standard Residues for a **Single** Protein Sequence
# @markdown After you input the sequence, click the run button
your_sequence = 'AAAXAAAAPPCCEUUUUUEECG' # @param {type:"string"}

import torch
import pandas as pd
from typing import Tuple


# table for standard amino acids
standard_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")


# use ESM model for predicting masked non-standard amino acids
model = EsmForMaskedLM.from_pretrained("facebook/esm2_t33_650M_UR50D")
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
model.eval()


def PLMezSeq(seq: str) -> str:
  """Predict non-standard residues in a single protein sequence."""
  masked_seq = mask_sequence(seq)

  if masked_seq == seq:
    return seq

  predicted_seq = predict_sequence(masked_seq)

  return predicted_seq


def mask_sequence(input_seq: str) -> str:
  """Replace non-standard amino acids with <mask>."""

  if len(input_seq) == 0:
    raise ValueError("Input sequence cannot be empty.")

  # find masked reisdues
  masked_res = []

  for aa in input_seq:
    if aa in standard_amino_acids:
      masked_res.append(aa)
    else:
      masked_res.append(tokenizer.mask_token)

  return "".join(masked_res)


def predict_sequence(masked_seq: str) -> str:
  """Use ESM2 to predict masked amino acids."""

  inputs = tokenizer(masked_seq, return_tensors="pt")

  with torch.no_grad():
    outputs = model(**inputs)

  logits = outputs.logits[0]
  input_ids = inputs["input_ids"][0]
  predicted_tokens = input_ids.clone()

  # find mask positions
  mask_positions = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]

  for pos in mask_positions:
    probabilities = torch.softmax(logits[pos], dim=-1)
    best_probability = -1
    best_aa = None

    # search only for standard amino acids
    for aa in standard_amino_acids:
      token_id = tokenizer.convert_tokens_to_ids(aa)
      probability = probabilities[token_id]

      if probability > best_probability:
        best_probability = probability
        best_aa = aa

    predicted_tokens[pos] = tokenizer.convert_tokens_to_ids(best_aa)

  # decode sequence
  output_seq = tokenizer.decode(predicted_tokens, skip_special_tokens=True)
  output_seq = output_seq.replace(" ", "")

  return output_seq


# call the main function
PLMezSeq(your_sequence)


Loading weights:   0%|          | 0/539 [00:00<?, ?it/s]

'AAAAAAAAPPCCEEEEEEEECG'

In [ ]:
# @title Predict Non-standard Residues for a **Batch** of Protein Sequence
# @markdown After you input the sequence, click the run button

import torch
import pandas as pd
from transformers import AutoTokenizer, EsmForMaskedLM
from typing import Tuple



In [11]:
import torch


test_sequences = [
  "MKTLLV<mask>GAAA",
  "ACDEFGHIK<mask>LMNPQRST",
  "M<mask>TA<mask>AA<mask>GG",
  "GILGFVFTL<mask>PSAQ"
]


def batch_predict_masked_sequences(masked_sequences: list[str], model, tokenizer, device) -> list[str]:
  """Predict multiple masked amino acids using ESM2 and return reconstructed sequences."""

  # set the model to evaluation mode
  model.eval()

  # send model to device
  model.to(device)

  # batch tokenize
  inputs = tokenizer(masked_sequences, return_tensors="pt", padding=True)

  inputs = {k: v.to(device) for k, v in inputs.items()}

  # find the positions of <mask> tokens in sequences
  mask_id = tokenizer.mask_token_id

  mask_positions = []

  # loop through each token list, each list corresponds to a tokenized sequence
  # mask positions is a list of lists, each inner list contains the positions of <mask> tokens in the corresponding sequence
  for ids in inputs["input_ids"]:

    positions = (ids == mask_id).nonzero(as_tuple=True)[0]

    mask_positions.append(positions.tolist())

  # Use ESM2 to predict the masked tokens
  with torch.no_grad():
    outputs = model(**inputs)

  logits = outputs.logits

  predicted_sequences = []

  # loop through each sequence and its corresponding mask positions
  for i, positions in enumerate(mask_positions):

    token_ids = inputs["input_ids"][i].clone()

    for pos in positions:

      # calculate probability distribution over vocabulary
      probabilities = torch.softmax(logits[i, pos], dim=-1)

      best_probability = -1
      best_token_id = None

      # only search standard amino acids
      for aa in standard_amino_acids:
        token_id = tokenizer.convert_tokens_to_ids(aa)
        probability = probabilities[token_id]

        if probability > best_probability:
          best_probability = probability
          best_token_id = token_id

      # replace <mask> token with predicted amino acid
      token_ids[pos] = best_token_id

    # convert tokens back to sequence
    sequence = tokenizer.decode(token_ids, skip_special_tokens=True)
    sequence = sequence.replace(" ", "")
    predicted_sequences.append(sequence)

  return predicted_sequences

# Run prediction

results = batch_predict_masked_sequences(test_sequences, model, tokenizer, device)


for seq in results:

  print(seq)

MKTLLVLGAAA
ACDEFGHIKLLMNPQRST
MATAAAAAGG
GILGFVFTLLPSAQ
